# Remote-point MPC with translation and rotation

This test uses two fields:

- `U`: translational displacement,
- `Φ`: scalar rotation about the out-of-plane `z` axis.

The physical displacement of the rigidly coupled end is

\[
u_{\mathrm{phys}} = u + R\varphi ,
\]

where `R = rigidRotationMap(mpc_u, mpc_φ)`.

The same kinematic map must be used consistently in the stiffness and mass
matrices. Consequently, both matrices have the block form

\[
\begin{bmatrix}
A & AR\\
R^T A & R^T A R
\end{bmatrix}.
\]

This allows the remote point to carry both concentrated forces and a
concentrated moment, and also gives the remote rotation the correct effective
rotational inertia in a dynamic problem.


In [1]:
using LowLevelFEM, LinearAlgebra

## Geometry and remote point


In [2]:
structured_rect_mesh(lx=10)

Premote = gmsh.model.occ.addPoint(10, 0.5, 0)
gmsh.model.occ.synchronize()
gmsh.model.addPhysicalGroup(0, [Premote], -1, "remote")

gmsh.model.mesh.clear()
gmsh.model.mesh.generate(3)

# openPreProcessor()

## Translational and rotational fields

The rotation field is scalar in 2D. It is only a generalized kinematic field;
its contribution to the physical displacement is introduced through `R`.


In [3]:
mat1 = Material("body")
mat2 = Material("remote")

U = Field(
    [mat1, mat2],
    type=:VectorField,
    dim=2,
    fieldName=:u,
    rhsName=:f
)

Φ = Field(
    [mat1, mat2],
    type=:ScalarField,
    dim=2,
    fieldName=:φ,
    rhsName=:m
)


Problem("structured_rect", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 1112, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false)

## Remote-point constraints and rigid-rotation map

`mpc_u` ties the translational part of every node on the right boundary to
the remote point. `mpc_φ` ties the rotation field to the remote rotation.
The actual rotation-induced displacement is generated by `R`.


In [4]:
mpc_u = MPC(
    master="remote",
    slave="right",
    field=U
)

mpc_φ = MPC(
    master="remote",
    slave="right",
    field=Φ
)

R = rigidRotationMap(mpc_u, mpc_φ);


## Stiffness matrix

If

\[
u_{\mathrm{phys}} = u + Rarphi,
\]

then the elastic energy gives the block stiffness matrix

\[
K =
egin{bmatrix}
K_u & K_uR\
R^TK_u & R^TK_uR
\end{bmatrix}.
\]


In [5]:
Ku = ∫(
    SymGrad(U) ⋅
    D(:PlaneStress, mat1) ⋅
    SymGrad(U),
    Ω="body"
)

Kuφ = Ku * R
Kφ  = R' * Ku * R

K = SystemMatrix([
    Ku    Kuφ
    Kuφ'  Kφ
]);


## Mass matrix

The kinetic energy must use the same physical velocity,

\[
\dot u_{\mathrm{phys}} = \dot u + R\dotarphi.
\]

Therefore the generalized mass matrix has exactly the same block structure.
This is essential for eigenvalue and transient dynamic calculations.


In [6]:
Mu = ∫(
    U ⋅ mat1.ρ ⋅ U,
    Ω="body"
)

Muφ = Mu * R
Mφ  = R' * Mu * R

M = SystemMatrix([
    Mu    Muφ
    Muφ'  Mφ
]);


## Boundary conditions and remote force/moment

The left edge is clamped in translation. The rotational field does not need
an independent boundary condition there: inactive rotational DOFs are removed
by the multifield/MPC reduction.

Both a force and a moment can be applied directly at the remote point.


In [7]:
bc = BoundaryCondition(
    "left",
    field=U,
    ux=0,
    uy=0
)

Fy = 1.0
Mz = 1.0

fu = ∫(
    U ⋅ [0, Fy],
    Γ="remote"
)

fφ = ∫(
    Φ ⋅ Mz,
    Γ="remote"
)

F = SystemVector([fu, fφ]);


## Static solution

`u` contains the translational generalized field and `φ` contains the
rotational generalized field. For visualization the physical displacement is

`u_phys = u + R * φ`.


In [8]:
u, φ = solveField(
    K,
    F;
    support=[bc],
    mpc=[mpc_u, mpc_φ]
)

u_phys = u + R * φ;


In [9]:
showDoFResults(
    u_phys,
    name="u",
    visible=true,
    factor=100
)


0

## Eigenmodes

The eigenproblem is solved for the coupled generalized system. The translational
and rotational parts are returned separately; the physical displacement mode
shape is reconstructed with the same rigid-rotation map.


In [24]:
u_modes, φ_modes = solveEigenFields(
    K,
    M;
    n=6,
    fmin=0.01,
    support=[bc],
    mpc=[mpc_u, mpc_φ]
)

u_phys_modes = LowLevelFEM.Eigen(
    u_modes.f,
    u_modes.ϕ + R.A * φ_modes.ϕ,
    U, nothing, nothing
);


In [25]:
showModalResults(u_phys_modes)


4

## Transient structural dynamics

At the moment the public `HHT` interface is single-field. The helper below is
a thin multifield/MPC wrapper around the already existing internal reduction
machinery and `_HHT_reduced` time integrator.

It is deliberately kept in the notebook so that the coupled remote-point
formulation can be tested before the public solver API is reorganized.


In [14]:
function HHT_multifield_mpc(
    K::SystemMatrix,
    M::SystemMatrix,
    F::SystemVector,
    X0::SystemVector,
    V0::SystemVector,
    n::Int,
    Δt::Float64;
    support=BoundaryCondition[],
    mpc::Vector{MPC}=MPC[],
    α=0.0,
    δ=0.0,
    γ=0.5 + δ,
    β=0.25 * (0.5 + γ)^2
    )

    isempty(mpc) &&
        error("HHT_multifield_mpc: this test helper expects at least one MPC.")

    LowLevelFEM.check_multifield_system_compatibility(K, M)

    K.problems == F.problems ||
        error("HHT_multifield_mpc: K and F use different field ordering.")

    K.problems == X0.problems ||
        error("HHT_multifield_mpc: K and X0 use different field ordering.")

    K.problems == V0.problems ||
        error("HHT_multifield_mpc: K and V0 use different field ordering.")

    K.offsets == F.offsets ||
        error("HHT_multifield_mpc: K and F use different offsets.")

    K.offsets == X0.offsets ||
        error("HHT_multifield_mpc: K and X0 use different offsets.")

    K.offsets == V0.offsets ||
        error("HHT_multifield_mpc: K and V0 use different offsets.")

    n > 1 ||
        error("HHT_multifield_mpc: n must be greater than one.")

    Δt > 0 ||
        error("HHT_multifield_mpc: Δt must be positive.")

    size(F.a, 2) == 1 || size(F.a, 2) == n ||
        error("HHT_multifield_mpc: F must contain either 1 or n time steps.")

    # ----------------------------------------------------------
    # 1) Global reduced-order + MPC transformation
    # ----------------------------------------------------------

    T, Rred, prunable, protected =
        LowLevelFEM._multifield_mpc_transformation(
            K,
            mpc
        )

    # ----------------------------------------------------------
    # 2) Full-space Dirichlet data
    # ----------------------------------------------------------

    _, fixed, xD =
        LowLevelFEM.multifield_bc_data(
            K,
            support;
            nsteps=n
        )

    # ----------------------------------------------------------
    # 3) Map BCs through MPC relations and reduced-order space
    # ----------------------------------------------------------

    fixed_mpc, xD_mpc =
        LowLevelFEM._multifield_mpc_bc_data(
            K,
            mpc,
            fixed,
            xD
        )

    free_r, fixed_r, _ =
        LowLevelFEM.reduced_bc_data(
            Rred,
            fixed_mpc,
            @view(xD_mpc[:, 1])
        )

    xD_r = Rred * xD_mpc

    # ----------------------------------------------------------
    # 4) Project system
    # ----------------------------------------------------------

    Kr = T' * K.A * T
    Mr = T' * M.A * T
    Fr = T' * F.a

    free_r =
        LowLevelFEM._remove_inactive_mpc_dofs(
            (Kr, Mr),
            Fr,
            free_r,
            fixed_r,
            protected,
            prunable
        )

    X0r = Rred * @view(X0.a[:, 1])
    V0r = Rred * @view(V0.a[:, 1])

    # ----------------------------------------------------------
    # 5) HHT integration in reduced space
    # ----------------------------------------------------------

    Xr, Vr =
        LowLevelFEM._HHT_reduced(
            Kr,
            Mr,
            Fr,
            X0r,
            V0r,
            xD_r,
            free_r,
            fixed_r,
            n,
            Δt;
            α=α,
            γ=γ,
            β=β
        )

    # ----------------------------------------------------------
    # 6) Prolongate complete histories
    # ----------------------------------------------------------

    X = T * Xr
    V = T * Vr

    t = collect(0:Δt:(n - 1) * Δt)

    Xfields =
        LowLevelFEM.split_multifield_solution(
            X,
            K.problems,
            K.offsets,
            t
        )

    Vfields =
        LowLevelFEM.split_multifield_solution(
            V,
            K.problems,
            K.offsets,
            t
        )

    return Xfields, Vfields
end


HHT_multifield_mpc (generic function with 1 method)

In [ ]:
function CDM_multifield_mpc(
    K::SystemMatrix,
    M::SystemMatrix,
    C::SystemMatrix,
    F::SystemVector,
    X0::SystemVector,
    V0::SystemVector,
    n::Int,
    Δt::Float64;
    support=BoundaryCondition[],
    mpc::Vector{MPC}=MPC[]
    )

    isempty(mpc) &&
        error("CDM_multifield_mpc: this helper expects at least one MPC.")

    check_multifield_system_compatibility(K, M)
    check_multifield_system_compatibility(K, C)

    K.problems == F.problems ||
        error("CDM_multifield_mpc: K and F use different field ordering.")

    K.problems == X0.problems ||
        error("CDM_multifield_mpc: K and X0 use different field ordering.")

    K.problems == V0.problems ||
        error("CDM_multifield_mpc: K and V0 use different field ordering.")

    K.offsets == F.offsets ||
        error("CDM_multifield_mpc: K and F use different offsets.")

    K.offsets == X0.offsets ||
        error("CDM_multifield_mpc: K and X0 use different offsets.")

    K.offsets == V0.offsets ||
        error("CDM_multifield_mpc: K and V0 use different offsets.")

    n > 1 ||
        error("CDM_multifield_mpc: n must be greater than one.")

    Δt > 0 ||
        error("CDM_multifield_mpc: Δt must be positive.")

    size(F.a, 2) == 1 || size(F.a, 2) == n ||
        error("CDM_multifield_mpc: F must contain either 1 or n time steps.")

    # ----------------------------------------------------------
    # 1) Reduced-order + MPC transformation
    # ----------------------------------------------------------

    T, Rred, prunable, protected =
        _multifield_mpc_transformation(
            K,
            mpc
        )

    # ----------------------------------------------------------
    # 2) Boundary conditions
    # ----------------------------------------------------------

    _, fixed, xD =
        multifield_bc_data(
            K,
            support;
            nsteps=n
        )

    fixed_mpc, xD_mpc =
        _multifield_mpc_bc_data(
            K,
            mpc,
            fixed,
            xD
        )

    free_r, fixed_r, _ =
        reduced_bc_data(
            Rred,
            fixed_mpc,
            @view(xD_mpc[:, 1])
        )

    xD_r = Rred * xD_mpc

    # ----------------------------------------------------------
    # 3) Project system
    # ----------------------------------------------------------

    Kr = T' * K.A * T
    Mr = T' * M.A * T
    Cr = T' * C.A * T
    Fr = T' * F.a

    free_r =
        _remove_inactive_mpc_dofs(
            (Kr, Mr, Cr),
            Fr,
            free_r,
            fixed_r,
            protected,
            prunable
        )

    X0r =
        Rred * @view(X0.a[:, 1])

    V0r =
        Rred * @view(V0.a[:, 1])

    # ----------------------------------------------------------
    # 4) CDM integration in reduced space
    # ----------------------------------------------------------

    Xr, Vr =
        _CDM_reduced(
            Kr,
            Mr,
            Cr,
            Fr,
            X0r,
            V0r,
            xD_r,
            free_r,
            fixed_r,
            n,
            Δt
        )

    # ----------------------------------------------------------
    # 5) Prolongate
    # ----------------------------------------------------------

    X = T * Xr
    V = T * Vr

    t =
        collect(
            0:Δt:(n - 1) * Δt
        )

    Xfields =
        split_multifield_solution(
            X,
            K.problems,
            K.offsets,
            t
        )

    Vfields =
        split_multifield_solution(
            V,
            K.problems,
            K.offsets,
            t
        )

    return Xfields, Vfields
end

In [ ]:
function CDM_multifield_mpc(
    K::SystemMatrix,
    M::SystemMatrix,
    F::SystemVector,
    X0::SystemVector,
    V0::SystemVector,
    n::Int,
    Δt::Float64;
    support=BoundaryCondition[],
    mpc::Vector{MPC}=MPC[]
    )

    C = K * 0.0
    dropzeros!(C.A)

    return CDM_multifield_mpc(
        K,
        M,
        C,
        F,
        X0,
        V0,
        n,
        Δt;
        support=support,
        mpc=mpc
    )
end

### Time step and initial conditions

The HHT method is implicit, so the value below is not a strict stability
limit. The largest eigenvalue is nevertheless useful for choosing a time step
that resolves the highest retained frequencies.


In [15]:
λmax = largestEigenValue(
    K,
    M;
    support=[bc],
    mpc=[mpc_u, mpc_φ]
)

Δtcrit = 2 / √λmax
Δt = Δtcrit / 5

nsteps = 300

X0 = SystemVector([
    0 * fu,
    0 * fφ
])

V0 = SystemVector([
    0 * fu,
    0 * fφ
]);


### Dynamic solution

A constant force and moment are applied from the first time step. The returned
generalized displacement histories are again combined into the physical
displacement field.


In [16]:
(x_hist, φ_hist), (v_hist, ω_hist) =
    HHT_multifield_mpc(
        K,
        M,
        F,
        X0,
        V0,
        nsteps,
        Δt;
        support=[bc],
        mpc=[mpc_u, mpc_φ],
        α=0.0
    )

u_hist = x_hist + R * φ_hist
v_phys_hist = v_hist + R * ω_hist;


In [17]:
showDoFResults(
    u_hist,
    name="u(t)",
    visible=true,
    factor=100
)


3

In [26]:
openPostProcessor()
